In [18]:
import mlrun

# Loads AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY and MLRUN_AWS_ROLE_ARN
from dotenv import load_dotenv
load_dotenv() 

from pathlib import Path
from datetime import datetime

artifact_path = Path.cwd().parent
artifact_path = str(artifact_path.as_posix()) # convert windows path to unix path
artifact_path = "file://" + artifact_path
p = mlrun.set_environment("http://localhost:8080", artifact_path=artifact_path)

project = mlrun.load_project(name='finetune-legal-extractor', context="../") # project yaml must be in this directory
# Verify it loaded correctly by checking its status or printing the config
# print(project.to_yaml())

In [19]:
system_prompt = """
You are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. You must respond ONLY with valid, minified JSON. Each hypothesis is a statement that may be supported, contradicted, or not mentioned based on the content of the contract. Read the whole contract and look for phrases or quotes in the contract to label each hypothesis. Label each hypothesis with:
"entailment" if the hypothesis is supported by the contract
"contradiction" if the hypothesis is contradicted by the contract
"not_mentioned" if the hypothesis cannot be confirmed or denied from the contract

Perform the analysis for all 17 hypotheses. The hypotheses with their corresponding ids are as follows:

nda-1: All Confidential Information shall be expressly identified by the Disclosing Party.
nda-2: Confidential Information shall only include technical information.
nda-3: Confidential Information may include verbally conveyed information.
nda-4: Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in the Agreement.
nda-5: Receiving Party may share some Confidential Information with some of Receiving Party's employees.
nda-7: Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors).
nda-8: Receiving Party shall notify Disclosing Party in case Receiving Party is required by law, regulation or judicial process to disclose any Confidential Information.
nda-10: Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.
nda-11: Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information.
nda-12: Receiving Party may independently develop information similar to Confidential Information.
nda-13: Receiving Party may acquire information similar to Confidential Information from a third party.
nda-15: Agreement shall not grant Receiving Party any right to Confidential Information.
nda-16: Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.
nda-17: Receiving Party may create a copy of some Confidential Information in some circumstances.
nda-18: Receiving Party shall not solicit some of Disclosing Party's representatives.
nda-19: Some obligations of Agreement may survive termination of Agreement.
nda-20: Receiving Party may retain some Confidential Information even after the return or destruction of Confidential Information.

Read the contract and respond with a JSON object that contains a single key-value pair; the key is "hypotheses", and its value is an array of 17 JSON objects for each hypothesis. Each object contains the id of the hypothesis, the hypothesis statement, the quotes or phrases from the contract that justifies the label given (this must only come from the legal contract from the user), and the label (entailment, contradiction, not_mentioned). Do not quote excessively and only quote the text that is relevant to the hypothesis that supports your decision of the label you assigned. For hypotheses that are labeled as not_mentioned leave the source_clause field blank. Only respond with the JSON document, do not provide any additional information, and do not make up your own quotes to fill in source_clause.

Remember:
If label is entailment or contradiction. Fill in source_clause with the quotes that justify the label even if they are from different locations
If label cannot be confirmed or denied label as "not_mentioned". Leave source_clause blank

Follow this format for each label
{ "hypotheses": 
    [
        {
            "hypothesis_id":...,
            "hypothesis":...,
            "source_clause": "Placeholder text",
            "label": "entailment"
        },
        {
            "hypothesis_id":...,
            "hypothesis":...,
            "source_clause": "Placeholder text",
            "label": "contradiction"
        },
        {
            "hypothesis_id":...,
            "hypothesis":...,
            "source_clause": "",
            "label": "not_mentioned"
        }
    ]
}
"""

In [20]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("JerroldK/Hermes-4-14B-contract-extractor")
print("Tokenizer loaded")
print(f"The number of tokens in the system prompt: {len(tok(system_prompt)['input_ids'])}")

Tokenizer loaded
The number of tokens in the system prompt: 837


### vLLM inference parameters
https://docs.djl.ai/master/docs/serving/serving/docs/lmi/user_guides/lmi_input_output_schema.html#additional-vllm-generation-parameters

In [21]:
# openAI 
prompt_template=[
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": "{contract}",
    }
]

# # chatml template
# prompt_template = [{
#     "template": (
#         "<|im_start|>system\n"
#         f"{system_prompt}<|im_end|>\n"
#         "<|im_start|>user\n"
#         "{contract}<|im_end|>\n"
#         "<|im_start|>assistant\n"
#     )
# }]


tag = datetime.now().strftime("%Y%m%d_%H%M") # this is the version
key = "contract_extractor_prompt"
print(tag)

project.log_llm_prompt(
    key=key,
    prompt_template=prompt_template,
    prompt_legend={
        "issue_description": {
            "field": "contract",
            "description": "The legal contract to extract data from",
        },
    },
    # https://docs.djl.ai/master/docs/serving/serving/docs/lmi/user_guides/lmi_input_output_schema.html#additional-vllm-generation-parameters
    invocation_config={
        "temperature": 0.2,
        "top_p": 0.95,
        "max_new_tokens": 3000, # default is extremely small. Lower to 3000
        "stop": ["<|im_end|>", "<|im_start|>"],
        "response_format": {"type": "json_object"} # This is works, but is not in docs
    },
    description="Prompt template for the legal extractor",
    artifact_path=f"s3://legal-llama-data/llm_prompt/{key}/{tag}",#artifact_path + f"/prompt/{key}/{tag}",
    tag=tag
)


20260508_1243


In [22]:
project.save(store=True)

In [23]:
# Retrieving the prompt
prompt = project.list_llm_prompts(name="contract_extractor_prompt", tag="latest")[0].read_prompt()
print(type(prompt[0]))
print(prompt)

<class 'dict'>
[{'role': 'system', 'content': '\nYou are a legal contract analyst assistant that analyzes a legal contract to confirm or deny the status of a set of hypotheses about the contract. You must respond ONLY with valid, minified JSON. Each hypothesis is a statement that may be supported, contradicted, or not mentioned based on the content of the contract. Read the whole contract and look for phrases or quotes in the contract to label each hypothesis. Label each hypothesis with:\n"entailment" if the hypothesis is supported by the contract\n"contradiction" if the hypothesis is contradicted by the contract\n"not_mentioned" if the hypothesis cannot be confirmed or denied from the contract\n\nPerform the analysis for all 17 hypotheses. The hypotheses with their corresponding ids are as follows:\n\nnda-1: All Confidential Information shall be expressly identified by the Disclosing Party.\nnda-2: Confidential Information shall only include technical information.\nnda-3: Confidential

In [6]:
project.get_artifact(key="contract_extractor_prompt", tag="latest").to_dict()['spec']['invocation_config']

{'temperature': 0.2,
 'top_p': 0.95,
 'max_new_tokens': 3000,
 'stop': ['<|im_end|>', '<|im_start|>'],
 'response_format': {'type': 'json_object'}}

In [ ]:
# get the id of a specific version
# version is specified with the tag which is "latest" or "yyyymmdd"
project.get_artifact(key="contract_extractor_prompt", tag="latest").to_dict()['spec']['producer']['tag']

'1ed287f80e848fc09bf73f3642b766d3887b8a83'

In [8]:
asdasd

NameError: name 'asdasd' is not defined

In [ ]:
# The only way to properly delete a data artifact and its historical versions
import os 

artifact = project.get_artifact(key="contract_extractor_prompt")
project.delete_artifact(artifact, 
                        deletion_strategy=mlrun.common.schemas.artifact.ArtifactsDeletionStrategies.data_force,
                        secrets={
                            "AWS_ACCESS_KEY_ID": os.environ['AWS_ACCESS_KEY_ID'],
                            "AWS_SECRET_ACCESS_KEY": os.environ['AWS_SECRET_ACCESS_KEY']
                            }
                        )